# Generate binary MIMIC-IV discharge documents

Fills `mimiciv_note.discharge_binary` with PDF and DOCX versions of discharge notes, for testing the OCR branch of NiFi flow 1.

1. Set `MODE` in the configuration cell:
   - `new_sample` empties the table and draws a new sample (`SAMPLE_FRACTION`, optionally fixed with `SEED`).
   - `same_notes` keeps the notes already in the table and rebuilds their files, each in its current format.
2. Run all cells (**Run → Run All Cells**).
3. In NiFi flow 1, clear the state of **GenerateTableFetch - MIMIC Notes** so it reads the rebuilt rows.

Running it again is always safe: rows are written with an upsert.

In [ ]:
import io
import random

import psycopg2
from docx import Document
from reportlab.lib.pagesizes import A4
from reportlab.lib.utils import simpleSplit
from reportlab.pdfgen import canvas

In [ ]:
# -----------------------------------------------
# CONFIGURATION
# -----------------------------------------------

DB_CONFIG = {
    "host": "samples-db",
    "dbname": "mimic",
    "user": "test",
    "password": "test",
    "port": 5432
}

# What to do with mimiciv_note.discharge_binary:
#   "new_sample"  - empty the table and fill it with a fresh sample of discharge notes
#   "same_notes"  - keep the notes already in the table and rebuild their files
#                   (each note keeps its current format, PDF or DOCX)
MODE = "new_sample"

SAMPLE_FRACTION = 0.05   # new_sample only: about 5% of discharge notes
SEED = None              # new_sample only: set an integer (e.g. 42) to get the same sample and formats every time

In [ ]:
# -----------------------------------------------
# HELPERS
# -----------------------------------------------

PDF_MIME = "application/pdf"
DOCX_MIME = "application/vnd.openxmlformats-officedocument.wordprocessingml.document"


def text_to_pdf(text: str, font: str = "Helvetica", size: int = 10) -> bytes:
    """Render the whole note: long lines wrap at the margin and new pages are
    added as needed (the old version wrote one page and cut off the rest)."""
    buffer = io.BytesIO()
    width, height = A4
    margin = 50
    leading = size * 1.25
    max_width = width - 2 * margin

    c = canvas.Canvas(buffer, pagesize=A4)

    def new_text_object():
        t = c.beginText(margin, height - margin)
        t.setFont(font, size)
        t.setLeading(leading)
        return t

    text_obj = new_text_object()
    y = height - margin
    for line in text.split("\n"):
        # simpleSplit returns [] for an empty line; keep blank lines as blank lines
        for part in simpleSplit(line, font, size, max_width) or [""]:
            if y < margin + leading:          # page full: start a new one
                c.drawText(text_obj)
                c.showPage()
                text_obj = new_text_object()
                y = height - margin
            text_obj.textLine(part)
            y -= leading

    c.drawText(text_obj)
    c.save()
    pdf_bytes = buffer.getvalue()
    buffer.close()
    return pdf_bytes


def text_to_docx(text: str) -> bytes:
    buffer = io.BytesIO()
    doc = Document()
    for line in text.split("\n"):
        doc.add_paragraph(line)
    doc.save(buffer)
    return buffer.getvalue()

In [ ]:
# -----------------------------------------------
# MAIN SCRIPT
# -----------------------------------------------

assert MODE in ("new_sample", "same_notes"), f"Unknown MODE {MODE!r}"
assert 0 < SAMPLE_FRACTION <= 1, "SAMPLE_FRACTION is a fraction: 0.05 means 5%"
rng = random.Random(SEED)

conn = psycopg2.connect(**DB_CONFIG)
with conn:                                   # one transaction: commits at the end, rolls back on error
    with conn.cursor() as cursor:
        if MODE == "new_sample":
            print(f"Emptying discharge_binary and sampling ~{SAMPLE_FRACTION:.0%} of discharge notes...")
            cursor.execute("TRUNCATE mimiciv_note.discharge_binary;")
            repeatable = f" REPEATABLE ({int(SEED)})" if SEED is not None else ""
            cursor.execute(f"""
                SELECT note_id, subject_id, hadm_id, note_type, note_seq,
                       charttime, storetime, text, NULL AS file_mime
                FROM mimiciv_note.discharge
                TABLESAMPLE SYSTEM ({SAMPLE_FRACTION * 100}){repeatable}
                ORDER BY note_id;
            """)
        else:
            print("Rebuilding the files for the notes already in discharge_binary...")
            cursor.execute("""
                SELECT d.note_id, d.subject_id, d.hadm_id, d.note_type, d.note_seq,
                       d.charttime, d.storetime, d.text, b.file_mime
                FROM mimiciv_note.discharge_binary b
                JOIN mimiciv_note.discharge d USING (note_id)
                ORDER BY d.note_id;
            """)
        rows = cursor.fetchall()
        print(f"Selected {len(rows)} notes.")

        counts = {PDF_MIME: 0, DOCX_MIME: 0}
        for i, (note_id, subject_id, hadm_id, note_type, note_seq,
                charttime, storetime, text, file_mime) in enumerate(rows, 1):

            # new_sample: random format; same_notes: keep the note's current format
            if file_mime not in (PDF_MIME, DOCX_MIME):
                file_mime = rng.choice([PDF_MIME, DOCX_MIME])
            binary_data = text_to_pdf(text) if file_mime == PDF_MIME else text_to_docx(text)
            counts[file_mime] += 1

            cursor.execute("""
                INSERT INTO mimiciv_note.discharge_binary (
                    note_id, subject_id, hadm_id, note_type, note_seq,
                    charttime, storetime, file_mime, file_data
                )
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
                ON CONFLICT (note_id) DO UPDATE SET
                    subject_id = EXCLUDED.subject_id,
                    hadm_id    = EXCLUDED.hadm_id,
                    note_type  = EXCLUDED.note_type,
                    note_seq   = EXCLUDED.note_seq,
                    charttime  = EXCLUDED.charttime,
                    storetime  = EXCLUDED.storetime,
                    file_mime  = EXCLUDED.file_mime,
                    file_data  = EXCLUDED.file_data;
            """, (
                note_id, subject_id, hadm_id, note_type, note_seq,
                charttime, storetime, file_mime, psycopg2.Binary(binary_data)
            ))
            if i % 1000 == 0:
                print(f"  {i} / {len(rows)}")
conn.close()

print(f"Done: {counts[PDF_MIME]} PDF and {counts[DOCX_MIME]} DOCX files in discharge_binary.")
print("Next: in NiFi flow 1, stop 'GenerateTableFetch - MIMIC Notes', View State -> Clear State, "
      "and start it again, so the documents are OCR'd again.")